
<div align="center">

# 🚀 AI Business Intelligence Copilot

### End-to-End Retail Analytics Platform with LLM-Powered Natural Language Insights

[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?logo=python&logoColor=white)](https://www.python.org/)
[![Pandas](https://img.shields.io/badge/Pandas-2.x-150458?logo=pandas&logoColor=white)](https://pandas.pydata.org/)
[![Scikit--Learn](https://img.shields.io/badge/Scikit--Learn-1.x-F7931E?logo=scikit-learn&logoColor=white)](https://scikit-learn.org/)
[![Prophet](https://img.shields.io/badge/Prophet-Forecasting-3F51B5)](https://facebook.github.io/prophet/)
[![Claude](https://img.shields.io/badge/Anthropic-Claude%20API-D97757?logo=anthropic&logoColor=white)](https://www.anthropic.com/)
[![License](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

</div>

---

## 📌 Project Overview

**AI Business Intelligence Copilot** is a production-style analytics notebook that fuses classic
**Business Intelligence (KPI dashboards, RFM segmentation, demand forecasting)** with a
**Large-Language-Model-powered natural language query layer**, inspired by well-known open-source
initiatives on GitHub and Kaggle such as:

| Reference Project | Concept Reused |
|---|---|
| [`pandas-ai`](https://github.com/gventuri/pandas-ai) | Conversational, LLM-driven DataFrame querying |
| [`vanna-ai`](https://github.com/vanna-ai/vanna) | Text-to-SQL / Text-to-Insight agent pattern |
| [`Online Retail II — UCI/Kaggle`](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci) | Real-world transactional retail dataset |
| [`lux`](https://github.com/lux-org/lux) | Automated, intelligent EDA visualization |
| [`superset`](https://github.com/apache/superset) | KPI dashboard design philosophy |

## 🗂️ Dataset

**Online Retail II** — a real, publicly available transactional dataset from a UK-based online
retailer, covering **01/12/2009 – 09/12/2011**, with invoices, product descriptions, quantities,
unit prices, customer IDs and countries.

- 🔗 UCI ML Repository: https://archive.ics.uci.edu/dataset/502/online+retail+ii
- 🔗 Kaggle mirror: https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci

## 🏗️ Architecture

```
Raw Transactional Data
        │
        ▼
Data Cleaning & Feature Engineering
        │
        ▼
 ┌──────────────┬───────────────────┬──────────────────┐
 │  KPI Engine   │  RFM Segmentation │  Demand Forecast  │
 └──────────────┴───────────────────┴──────────────────┘
        │
        ▼
 AI Copilot Layer (Claude API)
   • Natural-language → Pandas code
   • Automated executive summaries
        │
        ▼
   Exported Reports & Artifacts
```

## 📖 Table of Contents

1. [Environment Setup](#1)
2. [Configuration](#2)
3. [Data Acquisition](#3)
4. [Data Cleaning & Feature Engineering](#4)
5. [Exploratory Data Analysis](#5)
6. [KPI Dashboard](#6)
7. [Customer Segmentation (RFM + K-Means)](#7)
8. [Demand Forecasting (Prophet)](#8)
9. [AI Copilot — Natural Language Analytics](#9)
10. [Automated Executive Summary (LLM)](#10)
11. [Export Artifacts](#11)
12. [Conclusion & Next Steps](#12)

---


## 1. Environment Setup <a id='1'></a>

In [ ]:

%pip install -q pandas numpy matplotlib seaborn plotly scikit-learn prophet openpyxl kagglehub anthropic


## 2. Configuration <a id='2'></a>

In [ ]:

import os
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 6)

RANDOM_STATE = 42
CURRENCY_SYMBOL = "£"
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

@dataclass
class Config:
    dataset_kaggle_handle: str = "mashlyn/online-retail-ii-uci"
    dataset_uci_url: str = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
    raw_data_path: str = "data/online_retail_II.csv"
    output_dir: str = "outputs"
    snapshot_buffer_days: int = 1
    n_customer_segments: int = 4
    forecast_horizon_days: int = 90

CFG = Config()
os.makedirs(CFG.output_dir, exist_ok=True)
os.makedirs("data", exist_ok=True)


## 3. Data Acquisition <a id='3'></a>

Dataset source: **Online Retail II** ([Kaggle](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci) · [UCI](https://archive.ics.uci.edu/dataset/502/online+retail+ii)).

In [ ]:

def load_online_retail_ii(cfg: Config) -> pd.DataFrame:
    try:
        import kagglehub
        path = kagglehub.dataset_download(cfg.dataset_kaggle_handle)
        csv_candidates = [f for f in os.listdir(path) if f.lower().endswith(".csv")]
        df = pd.read_csv(os.path.join(path, csv_candidates[0]), encoding="ISO-8859-1")
    except Exception:
        sheets = pd.read_excel(
            "https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx",
            sheet_name=None,
        )
        df = pd.concat(sheets.values(), ignore_index=True)
    return df

df_raw = load_online_retail_ii(CFG)
df_raw.columns = [c.strip().replace(" ", "_") for c in df_raw.columns]
df_raw.head()


In [ ]:

print(f"Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]}")
df_raw.info()


## 4. Data Cleaning & Feature Engineering <a id='4'></a>

In [ ]:

def clean_transactions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename_map = {
        "Invoice": "InvoiceNo", "Customer_ID": "CustomerID",
        "Price": "UnitPrice", "InvoiceDate": "InvoiceDate",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    df = df.dropna(subset=["CustomerID"])
    df["CustomerID"] = df["CustomerID"].astype(int)
    df["InvoiceNo"] = df["InvoiceNo"].astype(str)
    df = df[~df["InvoiceNo"].str.startswith("C")]
    df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df["Revenue"] = df["Quantity"] * df["UnitPrice"]
    df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
    df["InvoiceDate_Day"] = df["InvoiceDate"].dt.date
    df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
    df["Hour"] = df["InvoiceDate"].dt.hour

    df = df.drop_duplicates()
    return df.reset_index(drop=True)

df = clean_transactions(df_raw)
print(f"Clean rows: {df.shape[0]:,} | Customers: {df['CustomerID'].nunique():,} | Countries: {df['Country'].nunique()}")
df.head()


## 5. Exploratory Data Analysis <a id='5'></a>

In [ ]:

monthly_revenue = df.groupby("InvoiceMonth", as_index=False)["Revenue"].sum()

fig = px.line(
    monthly_revenue, x="InvoiceMonth", y="Revenue",
    title="Monthly Revenue Trend", markers=True,
    labels={"InvoiceMonth": "Month", "Revenue": f"Revenue ({CURRENCY_SYMBOL})"},
)
fig.update_traces(line=dict(width=3))
fig.show()


In [ ]:

top_products = (
    df.groupby("Description", as_index=False)["Revenue"]
    .sum().sort_values("Revenue", ascending=False).head(15)
)

fig = px.bar(
    top_products.sort_values("Revenue"), x="Revenue", y="Description",
    orientation="h", title="Top 15 Products by Revenue",
    labels={"Revenue": f"Revenue ({CURRENCY_SYMBOL})", "Description": ""},
    color="Revenue", color_continuous_scale="Blues",
)
fig.show()


In [ ]:

country_revenue = (
    df.groupby("Country", as_index=False)["Revenue"]
    .sum().sort_values("Revenue", ascending=False).head(10)
)

fig = px.bar(
    country_revenue, x="Country", y="Revenue",
    title="Top 10 Countries by Revenue (excluding outliers)",
    labels={"Revenue": f"Revenue ({CURRENCY_SYMBOL})"},
    color="Revenue", color_continuous_scale="Teal",
)
fig.show()


In [ ]:

heatmap_data = (
    df.groupby(["DayOfWeek", "Hour"])["Revenue"].sum().unstack(fill_value=0)
)
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heatmap_data = heatmap_data.reindex(day_order)

plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, cmap="YlGnBu", linewidths=0.3)
plt.title("Revenue Intensity Heatmap — Day of Week vs Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Day of Week")
plt.tight_layout()
plt.show()


## 6. KPI Dashboard <a id='6'></a>

In [ ]:

def compute_kpis(df: pd.DataFrame) -> dict:
    total_revenue = df["Revenue"].sum()
    total_orders = df["InvoiceNo"].nunique()
    total_customers = df["CustomerID"].nunique()
    aov = total_revenue / total_orders
    revenue_per_customer = total_revenue / total_customers

    monthly = df.groupby("InvoiceMonth")["Revenue"].sum().sort_index()
    mom_growth = monthly.pct_change().iloc[-1] * 100 if len(monthly) > 1 else np.nan

    return {
        "Total Revenue": f"{CURRENCY_SYMBOL}{total_revenue:,.0f}",
        "Total Orders": f"{total_orders:,}",
        "Unique Customers": f"{total_customers:,}",
        "Average Order Value": f"{CURRENCY_SYMBOL}{aov:,.2f}",
        "Revenue per Customer": f"{CURRENCY_SYMBOL}{revenue_per_customer:,.2f}",
        "Latest Month-over-Month Growth": f"{mom_growth:,.1f}%",
    }

kpis = compute_kpis(df)
kpi_df = pd.DataFrame(list(kpis.items()), columns=["KPI", "Value"])
kpi_df


In [ ]:

fig = go.Figure()
labels = list(kpis.keys())[:4]
for i, label in enumerate(labels):
    fig.add_trace(go.Indicator(
        mode="number",
        value=0,
        title={"text": label},
        domain={"row": 0, "column": i},
    ))
fig.update_layout(grid={"rows": 1, "columns": 4, "pattern": "independent"}, height=220)

fig2 = px.bar(kpi_df, x="KPI", y=[0]*len(kpi_df), text="Value", title="Key Performance Indicators — Snapshot")
fig2.update_traces(textposition="outside")
fig2.update_yaxes(visible=False)
fig2.show()


## 7. Customer Segmentation — RFM + K-Means <a id='7'></a>

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=CFG.snapshot_buffer_days)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum"),
).reset_index()

rfm_log = rfm.copy()
for col in ["Recency", "Frequency", "Monetary"]:
    rfm_log[col] = np.log1p(rfm_log[col])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log[["Recency", "Frequency", "Monetary"]])

kmeans = KMeans(n_clusters=CFG.n_customer_segments, random_state=RANDOM_STATE, n_init=10)
rfm["Segment"] = kmeans.fit_predict(rfm_scaled)

segment_profile = rfm.groupby("Segment").agg(
    Customers=("CustomerID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
).round(1).sort_values("Avg_Monetary", ascending=False)

segment_names = {seg: name for seg, name in zip(
    segment_profile.index,
    ["Champions", "Loyal Customers", "At Risk", "Hibernating"][:len(segment_profile)]
)}
rfm["Segment_Name"] = rfm["Segment"].map(segment_names)
segment_profile["Segment_Name"] = segment_profile.index.map(segment_names)
segment_profile


In [ ]:

fig = px.scatter_3d(
    rfm, x="Recency", y="Frequency", z="Monetary",
    color="Segment_Name", title="Customer Segments — RFM Space",
    opacity=0.7,
)
fig.show()


In [ ]:

segment_counts = rfm["Segment_Name"].value_counts().reset_index()
segment_counts.columns = ["Segment", "Customers"]

fig = px.pie(
    segment_counts, names="Segment", values="Customers",
    title="Customer Base Composition by Segment", hole=0.45,
)
fig.show()


## 8. Demand Forecasting — Prophet <a id='8'></a>

In [ ]:

from prophet import Prophet

daily_revenue = (
    df.groupby("InvoiceDate_Day", as_index=False)["Revenue"].sum()
    .rename(columns={"InvoiceDate_Day": "ds", "Revenue": "y"})
)
daily_revenue["ds"] = pd.to_datetime(daily_revenue["ds"])

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.1,
)
model.fit(daily_revenue)

future = model.make_future_dataframe(periods=CFG.forecast_horizon_days)
forecast = model.predict(future)


In [ ]:

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_revenue["ds"], y=daily_revenue["y"], mode="lines", name="Actual"))
fig.add_trace(go.Scatter(x=forecast["ds"], y=forecast["yhat"], mode="lines", name="Forecast"))
fig.add_trace(go.Scatter(
    x=pd.concat([forecast["ds"], forecast["ds"][::-1]]),
    y=pd.concat([forecast["yhat_upper"], forecast["yhat_lower"][::-1]]),
    fill="toself", fillcolor="rgba(99,110,250,0.15)", line=dict(width=0),
    name="Confidence Interval",
))
fig.update_layout(title=f"Revenue Forecast — Next {CFG.forecast_horizon_days} Days", xaxis_title="Date", yaxis_title=f"Revenue ({CURRENCY_SYMBOL})")
fig.show()


In [ ]:

fig_components = model.plot_components(forecast)
plt.show()


## 9. AI Copilot — Natural Language Analytics <a id='9'></a>

A lightweight text-to-pandas agent, in the spirit of `pandas-ai` and `vanna-ai`, powered by the Anthropic Claude API.

In [ ]:

import re
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

COPILOT_SYSTEM_PROMPT = '''You are a senior data analyst. You are given a pandas DataFrame `df`
with columns: {columns}. Convert the user's question into a single valid pandas expression that
computes the answer and assign it to a variable named `result`. Return ONLY a python code block,
no explanation.'''

def ask_copilot(question: str, dataframe: pd.DataFrame) -> dict:
    system_prompt = COPILOT_SYSTEM_PROMPT.format(columns=list(dataframe.columns))
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=400,
        system=system_prompt,
        messages=[{"role": "user", "content": question}],
    )
    raw_text = "".join(block.text for block in response.content if block.type == "text")
    code_match = re.search(r"```(?:python)?\s*(.*?)```", raw_text, re.DOTALL)
    generated_code = code_match.group(1).strip() if code_match else raw_text.strip()

    local_scope = {"df": dataframe, "pd": pd, "np": np}
    exec(generated_code, {}, local_scope)
    return {"question": question, "code": generated_code, "result": local_scope.get("result")}

sample_questions = [
    "What is the total revenue for the United Kingdom?",
    "Which 5 customers generated the highest total revenue?",
    "What is the average order value by country, top 5?",
]

for q in sample_questions:
    if ANTHROPIC_API_KEY:
        answer = ask_copilot(q, df)
        print(f"Q: {answer['question']}\nCode: {answer['code']}\nResult:\n{answer['result']}\n{'-'*60}")
    else:
        print(f"Q: {q}\n[Set ANTHROPIC_API_KEY to enable live Copilot responses]\n{'-'*60}")


## 10. Automated Executive Summary (LLM) <a id='10'></a>

In [ ]:

def generate_executive_summary(kpis: dict, segment_profile: pd.DataFrame) -> str:
    prompt = f'''Write a concise, professional executive summary (max 200 words) for a retail
business, based on these KPIs: {kpis}
and this customer segmentation profile: {segment_profile.to_dict()}
Focus on actionable insights and risks.'''

    if not ANTHROPIC_API_KEY:
        return "[Set ANTHROPIC_API_KEY to generate a live executive summary]"

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(block.text for block in response.content if block.type == "text")

executive_summary = generate_executive_summary(kpis, segment_profile)
print(executive_summary)


## 11. Export Artifacts <a id='11'></a>

In [ ]:

df.to_csv(f"{CFG.output_dir}/cleaned_transactions.csv", index=False)
rfm.to_csv(f"{CFG.output_dir}/customer_rfm_segments.csv", index=False)
forecast.to_csv(f"{CFG.output_dir}/revenue_forecast.csv", index=False)
kpi_df.to_csv(f"{CFG.output_dir}/kpi_summary.csv", index=False)

with open(f"{CFG.output_dir}/executive_summary.txt", "w") as f:
    f.write(executive_summary)

print("Artifacts exported to:", os.path.abspath(CFG.output_dir))
os.listdir(CFG.output_dir)



## 12. Conclusion & Next Steps <a id='12'></a>

This notebook delivers a full **BI + AI Copilot** pipeline on a real-world transactional dataset:

- ✅ Clean, reproducible ETL pipeline
- ✅ KPI dashboard covering revenue, orders, AOV and growth
- ✅ RFM-based customer segmentation with K-Means clustering
- ✅ Time-series demand forecasting with Prophet
- ✅ LLM-powered natural language analytics layer (Claude API)
- ✅ Automated executive summary generation

### Suggested Extensions

- Deploy the Copilot layer behind a **Streamlit** or **Gradio** app for interactive querying
- Add **SQL** connectivity (e.g., DuckDB) for the text-to-SQL variant of the Copilot
- Integrate **cohort retention analysis** and **churn prediction**
- Schedule automated weekly executive summaries via a cron job / Airflow DAG

---

<div align="center">

**Built with pandas · scikit-learn · Prophet · Plotly · Anthropic Claude**

</div>
